In [ ]:
# ============================================================
# IMPORT LIBRARY
# ============================================================

import pandas as pd
import numpy as np
from IPython.display import display, HTML

In [ ]:
# ============================================================
# LOAD DATA
# ============================================================

df_mahasiswa = pd.read_excel('/content/PBL_DATA_preprocessed.xlsx', sheet_name='MAHASISWA')
df_perusahaan = pd.read_excel('/content/PBL_DATA_preprocessed.xlsx', sheet_name='PERUSAHAAN')

print(f"Data mahasiswa  : {len(df_mahasiswa)} baris")
print(f"Data perusahaan : {len(df_perusahaan)} baris")

Data mahasiswa  : 305 baris
Data perusahaan : 30 baris


In [ ]:
# ============================================================
# TENTUKAN NIM MAHASISWA
# ============================================================

NIM_MAHASISWA = '848822'

# Ambil data mahasiswa berdasarkan NIM
mahasiswa = df_mahasiswa[df_mahasiswa['NIM'].astype(str) == NIM_MAHASISWA].iloc[0]

print(f"Nama  : {mahasiswa['Nama']}")
print(f"NIM   : {mahasiswa['NIM']}")
print(f"IPK   : {mahasiswa['IPK']}")
print(f"Skill : {mahasiswa['Skill']}")
print(f"Minat : {mahasiswa['Minat_Bidang']}")

Nama  : Dinda SIta
NIM   : 848822
IPK   : 3.61
Skill : Python, JavaScript, TypeScript, Java, SQL, Tailwind CSS, Bootstrap, HTML & CSS, MySQL, Git & GitHub, Figma, Canva, UI/UX Design
Minat : Software Developer, Artificial Intelligence, Data Technology, Multimedia and game, Information System, Business analyst


In [ ]:
# ============================================================
# PREPROCESSING: UBAH TEKS MENJADI SET
# ============================================================

def teks_ke_set(teks):
    """Ubah string 'A, B, C' menjadi {'a', 'b', 'c'}"""
    if pd.isna(teks) or str(teks).strip() == '':
        return set()
    return {item.strip().lower() for item in str(teks).split(',') if item.strip()}

# Terapkan ke seluruh baris mahasiswa
df_mahasiswa['skills_set']    = df_mahasiswa['Skill'].apply(teks_ke_set)
df_mahasiswa['teknologi_set'] = df_mahasiswa['Teknologi'].apply(teks_ke_set)
df_mahasiswa['minat_set']     = df_mahasiswa['Minat_Bidang'].apply(teks_ke_set)

# Ambil ulang data mahasiswa setelah preprocessing
mahasiswa = df_mahasiswa[df_mahasiswa['NIM'].astype(str) == NIM_MAHASISWA].iloc[0]

print("Preprocessing selesai.")
print(f"Jumlah skill    : {len(mahasiswa['skills_set'])}")
print(f"Jumlah teknologi: {len(mahasiswa['teknologi_set'])}")
print(f"Jumlah minat    : {len(mahasiswa['minat_set'])}")

Preprocessing selesai.
Jumlah skill    : 13
Jumlah teknologi: 16
Jumlah minat    : 6


In [ ]:
# ============================================================
# RUMUS KOMBINASI CERTAINTY FACTOR
#
# Rumus standar Shortliffe & Buchanan (1975):
#   Jika CF1 dan CF2 keduanya positif → CF1 + CF2*(1-CF1)
#   Jika CF1 dan CF2 keduanya negatif → CF1 + CF2*(1+CF1)
#   Jika berbeda tanda                → (CF1+CF2) / (1-min(|CF1|,|CF2|))
# Hasil selalu berada di antara -1.0 dan 1.0
# ============================================================

def gabung_dua_cf(cf1, cf2):
    """Gabungkan dua nilai CF menjadi satu."""
    if cf1 >= 0 and cf2 >= 0:
        return cf1 + cf2 * (1 - cf1)
    elif cf1 <= 0 and cf2 <= 0:
        return cf1 + cf2 * (1 + cf1)
    else:
        penyebut = 1 - min(abs(cf1), abs(cf2))
        if penyebut == 0:
            return cf1 + cf2
        return (cf1 + cf2) / penyebut

def gabung_semua_cf(daftar_cf):
    """Gabungkan daftar CF secara berurutan (satu per satu)."""
    hasil = float(daftar_cf[0])
    for cf in daftar_cf[1:]:
        hasil = gabung_dua_cf(hasil, float(cf))
    # Pastikan hasil tidak melampaui batas -1 hingga 1
    return round(max(min(hasil, 1.0), -1.0), 4)

print("Fungsi kombinasi CF siap digunakan.")

Fungsi kombinasi CF siap digunakan.


In [ ]:
# ============================================================
# HITUNG CF UNTUK SETIAP FAKTOR
#
# Ada 5 faktor yang dievaluasi:
#   1. CF_IPK        : apakah IPK mahasiswa memenuhi syarat?
#   2. CF_Skill      : seberapa banyak skill yang cocok?
#   3. CF_Tools      : seberapa banyak teknologi yang cocok?
#   4. CF_Minat      : apakah minat sesuai posisi/bidang?
#   5. CF_Preferensi : apakah status paid sesuai preferensi?
#
# Setiap CF bernilai antara -1.0 (sangat tidak cocok) hingga +1.0 (sangat cocok)
# ============================================================

# ------ Bobot masing-masing faktor (total = 1.0) ----------
BOBOT_IPK        = 0.25
BOBOT_SKILL      = 0.30
BOBOT_TOOLS      = 0.20
BOBOT_MINAT      = 0.15
BOBOT_PREFERENSI = 0.10


# ---- Faktor 1: IPK ----------------------------------------
# Semakin jauh IPK di atas syarat → CF semakin tinggi
# IPK di bawah syarat → CF negatif

def hitung_cf_ipk(ipk_mahasiswa, ipk_minimal):
    if pd.isna(ipk_minimal) or float(ipk_minimal) == 0:
        return 0.50   # Tidak ada syarat IPK → netral
    selisih = ipk_mahasiswa - float(ipk_minimal)
    if selisih >= 0.5:  return  0.95   # Jauh di atas syarat
    elif selisih >= 0.2: return  0.85
    elif selisih >= 0.0: return  0.70   # Tepat memenuhi syarat
    elif selisih >= -0.1: return -0.30  # Sedikit di bawah syarat
    elif selisih >= -0.3: return -0.65
    else:               return -0.90   # Jauh di bawah syarat


# ---- Faktor 2: Skill --------------------------------------
# Hitung berapa persen skill perusahaan yang dimiliki mahasiswa

def hitung_cf_skill(skills_mahasiswa, skill_dibutuhkan_teks):
    skill_perusahaan = teks_ke_set(skill_dibutuhkan_teks)
    if not skill_perusahaan:
        return 0.50, 0, 0.0   # Tidak ada data → netral
    cocok  = skills_mahasiswa & skill_perusahaan
    rasio  = len(cocok) / len(skill_perusahaan)
    persen = round(rasio * 100, 2)
    if rasio >= 0.80:  cf = 0.95
    elif rasio >= 0.60: cf = 0.80
    elif rasio >= 0.40: cf = 0.60
    elif rasio >= 0.20: cf = 0.35
    elif rasio > 0.0:   cf = 0.15
    else:               cf = -0.50  # Tidak ada yang cocok sama sekali
    return cf, len(cocok), persen


# ---- Faktor 3: Tools/Teknologi ----------------------------
# Hitung berapa persen teknologi perusahaan yang dikuasai mahasiswa

def hitung_cf_tools(teknologi_mahasiswa, teknologi_perusahaan_teks):
    tools_perusahaan = teks_ke_set(teknologi_perusahaan_teks)
    if not tools_perusahaan:
        return 0.50, 0, 0.0
    cocok  = teknologi_mahasiswa & tools_perusahaan
    rasio  = len(cocok) / len(tools_perusahaan)
    persen = round(rasio * 100, 2)
    if rasio >= 0.70:  cf = 0.90
    elif rasio >= 0.50: cf = 0.75
    elif rasio >= 0.30: cf = 0.55
    elif rasio >= 0.10: cf = 0.30
    elif rasio > 0.0:   cf = 0.10
    else:               cf = -0.40
    return cf, len(cocok), persen


# ---- Faktor 4: Minat Bidang -------------------------------
# Cek apakah minat mahasiswa muncul di teks posisi/deskripsi perusahaan

def hitung_cf_minat(minat_mahasiswa, posisi, job_desc, bidang):
    if not minat_mahasiswa:
        return 0.30
    teks = (str(posisi) + ' ' + str(job_desc) + ' ' + str(bidang)).lower()
    jumlah_cocok = sum(1 for m in minat_mahasiswa if m in teks)
    rasio = jumlah_cocok / len(minat_mahasiswa)
    if rasio >= 0.60:  return 0.85
    elif rasio >= 0.30: return 0.65
    elif rasio > 0.0:   return 0.45
    else:               return 0.10   # Tidak ada minat yang cocok


# ---- Faktor 5: Preferensi Paid/Unpaid ---------------------
# Preferensi otomatis: lebih baik Paid, tapi Unpaid tetap diterima
# Logika: Paid = 1.0 (prioritas), Unpaid = 0.5 (tidak masalah), Flexible = 0.8

def hitung_cf_preferensi(status_paid_perusahaan):
    status = str(status_paid_perusahaan).strip().lower()
    if status == 'paid':     return 0.90   # Ideal → dibayar
    elif status == 'flexible': return 0.80   # Bisa paid atau unpaid
    elif status == 'unpaid':  return 0.50   # Tidak masalah, tapi bukan prioritas
    else:                     return 0.50   # Status tidak diketahui → netral


print("Semua fungsi CF per faktor siap.")

Semua fungsi CF per faktor siap.


In [ ]:
# ============================================================
# HITUNG CF TOTAL UNTUK SEMUA PERUSAHAAN
#
# Untuk setiap perusahaan, hitung kelima CF di atas,
# kalikan dengan bobotnya, lalu gabungkan menjadi CF total.
# ============================================================

hasil_semua = []   # Akan menampung hasil CF untuk semua perusahaan

for _, perusahaan in df_perusahaan.iterrows():

    # --- Hitung tiap faktor ---
    cf_ipk = hitung_cf_ipk(mahasiswa['IPK'], perusahaan['Minimal_IPK'])

    cf_skill, jml_skill, pct_skill = hitung_cf_skill(
        mahasiswa['skills_set'],
        perusahaan['Skill_Dibutuhkan']
    )

    cf_tools, jml_tools, pct_tools = hitung_cf_tools(
        mahasiswa['teknologi_set'],
        perusahaan['Teknologi_Digunakan']
    )

    cf_minat = hitung_cf_minat(
        mahasiswa['minat_set'],
        perusahaan['Posisi_Magang'],
        perusahaan['Job_Description'],
        perusahaan['Bidang_Industri']
    )

    cf_pref = hitung_cf_preferensi(perusahaan['Status_Paid'])

    # --- Kalikan CF dengan bobot masing-masing ---
    cf_berbobot = [
        cf_ipk   * BOBOT_IPK,
        cf_skill * BOBOT_SKILL,
        cf_tools * BOBOT_TOOLS,
        cf_minat * BOBOT_MINAT,
        cf_pref  * BOBOT_PREFERENSI,
    ]

    # --- Gabungkan semua CF berbobot menjadi satu nilai akhir ---
    cf_total = gabung_semua_cf(cf_berbobot)

    # --- Tentukan label rekomendasi berdasarkan nilai CF ---
    if cf_total >= 0.75:   label = '✅ Sangat Direkomendasikan'
    elif cf_total >= 0.50: label = '🟢 Direkomendasikan'
    elif cf_total >= 0.25: label = '🟡 Cukup Sesuai'
    elif cf_total >= 0.00: label = '🟠 Kurang Sesuai'
    else:                  label = '🔴 Tidak Direkomendasikan'

    # --- Simpan hasil ---
    hasil_semua.append({
        'ID'            : perusahaan['ID_Perusahaan'],
        'Perusahaan'    : perusahaan['Nama_Perusahaan'],
        'Posisi'        : perusahaan['Posisi_Magang'],
        'Status_Paid'   : perusahaan['Status_Paid'],
        'Durasi_Bulan'  : perusahaan['Durasi_Bulan'],
        'Min_IPK'       : perusahaan['Minimal_IPK'],
        'CF_IPK'        : round(cf_ipk,   4),
        'CF_Skill'      : round(cf_skill, 4),
        'CF_Tools'      : round(cf_tools, 4),
        'CF_Minat'      : round(cf_minat, 4),
        'CF_Preferensi' : round(cf_pref,  4),
        'CF_Total'      : cf_total,
        'Label'         : label,
        'Skill_Cocok'   : jml_skill,
        'Pct_Skill_%'   : pct_skill,
        'Tools_Cocok'   : jml_tools,
        'Pct_Tools_%'   : pct_tools,
    })

# Jadikan DataFrame dan urutkan dari CF tertinggi ke terendah
df_hasil = pd.DataFrame(hasil_semua)
df_hasil = df_hasil.sort_values('CF_Total', ascending=False).reset_index(drop=True)
df_hasil.index += 1   # Ranking mulai dari 1 (bukan 0)

print(f"Perhitungan selesai. Total perusahaan dievaluasi: {len(df_hasil)}")

Perhitungan selesai. Total perusahaan dievaluasi: 30


In [ ]:
# ============================================================
# MENAMPILKAN PROFIL MAHASISWA
# ============================================================

print("=" * 60)
print("  PROFIL MAHASISWA")
print("=" * 60)
print(f"  Nama      : {mahasiswa['Nama']}")
print(f"  NIM       : {mahasiswa['NIM']}")
print(f"  IPK       : {mahasiswa['IPK']}")
print(f"  Preferensi: Paid lebih baik, Unpaid tetap diterima")
print(f"  Skill     : {mahasiswa['Skill'][:80]}...")
print(f"  Minat     : {mahasiswa['Minat_Bidang']}")
print("=" * 60)

  PROFIL MAHASISWA
  Nama      : Dinda SIta
  NIM       : 848822
  IPK       : 3.61
  Preferensi: Paid lebih baik, Unpaid tetap diterima
  Skill     : Python, JavaScript, TypeScript, Java, SQL, Tailwind CSS, Bootstrap, HTML & CSS, ...
  Minat     : Software Developer, Artificial Intelligence, Data Technology, Multimedia and game, Information System, Business analyst


In [ ]:
# ============================================================
# MENAMPILKAN TOP 5 REKOMENDASI SEBAGAI TABEL
# ============================================================

top5 = df_hasil.head(5)[['Perusahaan', 'Posisi', 'Status_Paid',
                           'CF_Total', 'Label',
                           'CF_IPK', 'CF_Skill', 'CF_Tools', 'CF_Minat', 'CF_Preferensi',
                           'Pct_Skill_%', 'Pct_Tools_%']].copy()

print("\nTOP 5 REKOMENDASI TEMPAT MAGANG (Certainty Factor)\n")
display(top5)


TOP 5 REKOMENDASI TEMPAT MAGANG (Certainty Factor)



,Perusahaan,Posisi,Status_Paid,CF_Total,Label,CF_IPK,CF_Skill,CF_Tools,CF_Minat,CF_Preferensi,Pct_Skill_%,Pct_Tools_%
1,PT Peruri Wira Timur,Backend/Fullstack Develope...,Paid,0.4250,🟡 Cukup Sesuai,0.95,0.35,0.30,0.10,0.9,22.22,28.57
2,PT Industri Kereta Api / P...,Software Developer / Data ...,Paid,0.4192,🟡 Cukup Sesuai,0.95,0.15,0.30,0.45,0.9,12.50,14.29
3,Ariverse Studio (PT Studio...,Game Programmer Intern / F...,Paid,0.4191,🟡 Cukup Sesuai,0.95,0.15,0.55,0.10,0.9,10.00,37.50
4,CV DB KLIK,Backend/Frontend Developer...,Paid,0.4191,🟡 Cukup Sesuai,0.95,0.15,0.55,0.10,0.9,14.29,33.33
5,Humas POLINEMA,Data Analyst / Social Medi...,Unpaid,0.3997,🟡 Cukup Sesuai,0.95,0.35,0.30,0.10,0.5,20.00,16.67


In [ ]:
# ============================================================
# MENAMPILKAN ANALISIS DETAIL PERINGKAT #1
# ============================================================

top1 = df_hasil.iloc[0]

print("\n" + "=" * 60)
print("  ANALISIS DETAIL – PERINGKAT #1")
print("=" * 60)
print(f"  Perusahaan  : {top1['Perusahaan']}")
print(f"  Posisi      : {top1['Posisi']}")
print(f"  Status Paid : {top1['Status_Paid']}")
print(f"  Durasi      : {top1['Durasi_Bulan']} bulan")
print(f"  Min. IPK    : {top1['Min_IPK']}")
print()
print(f"  CF TOTAL    : {top1['CF_Total']}  →  {top1['Label']}")
print()
print(f"  ┌ CF IPK        : {top1['CF_IPK']:.4f}")
print(f"  │  IPK mahasiswa {mahasiswa['IPK']} vs syarat minimal {top1['Min_IPK']}")
print()
print(f"  ├ CF Skill      : {top1['CF_Skill']:.4f}")
print(f"  │  {top1['Skill_Cocok']} skill cocok ({top1['Pct_Skill_%']}% dari skill yang dibutuhkan)")
print()
print(f"  ├ CF Tools      : {top1['CF_Tools']:.4f}")
print(f"  │  {top1['Tools_Cocok']} tools cocok ({top1['Pct_Tools_%']}% dari tools perusahaan)")
print()
print(f"  ├ CF Minat      : {top1['CF_Minat']:.4f}")
print(f"  │  Minat mahasiswa dicocokkan dengan posisi & deskripsi pekerjaan")
print()
print(f"  └ CF Preferensi : {top1['CF_Preferensi']:.4f}")
print(f"     Status perusahaan: {top1['Status_Paid']}")


  ANALISIS DETAIL – PERINGKAT #1
  Perusahaan  : PT Peruri Wira Timur
  Posisi      : Backend/Fullstack Developer – Sistem Internal
  Status Paid : Paid
  Durasi      : 6 bulan
  Min. IPK    : 3

  CF TOTAL    : 0.425  →  🟡 Cukup Sesuai

  ┌ CF IPK        : 0.9500
  │  IPK mahasiswa 3.61 vs syarat minimal 3

  ├ CF Skill      : 0.3500
  │  2 skill cocok (22.22% dari skill yang dibutuhkan)

  ├ CF Tools      : 0.3000
  │  2 tools cocok (28.57% dari tools perusahaan)

  ├ CF Minat      : 0.1000
  │  Minat mahasiswa dicocokkan dengan posisi & deskripsi pekerjaan

  └ CF Preferensi : 0.9000
     Status perusahaan: Paid
